In [ ]:
import os
import time

from data_generation import generate_design, generate_response
from SI.Practical_TM_SI import practical_TM_SI_randj
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

In [ ]:
p, nt, ns, K = 100, 30, 50, 2
V = 3
sigma = 1.0
n_rep = 1000
#n_jobs = os.cpu_count()   # use every logical core
n_jobs = 12

In [ ]:
print(n_jobs)

In [ ]:
rng = np.random.default_rng(185)

X0, X_list = generate_design(rng, p, nt, ns, K)

Sigma_0 = sigma ** 2 * np.eye(nt)
Sigma_K = [sigma ** 2 * np.eye(ns) for _ in range(K)]

In [ ]:
p_values, branches, seconds = [], [], []  
kkt = {"observed fits": 0, "observed violations": 0, "SI fits": 0, "SI violations": 0}
invalid = []   # (iteration, p-value) of non-finite or out-of-range p-values, left out


def run(n_rep):
    n_empty = 0
    start = time.perf_counter()

    for i in range(1, n_rep + 1):
        y0, y_list = generate_response(rng, X0, X_list, sigma)

        t = time.perf_counter()
        result = practical_TM_SI_randj(
            X0, y0, X_list, y_list, Sigma_0, Sigma_K,
            V=V, rng=rng, n_jobs=n_jobs,
        )

        kkt["observed fits"] += result['kkt_observed']['fits']
        kkt["observed violations"] += result['kkt_observed']['violations']
        if result['results']:
            p_value = result['results'][0]['p_value']
            if np.isfinite(p_value) and 0.0 <= p_value <= 1.0:
                p_values.append(p_value)
                branches.append(result['branch'])
                seconds.append(time.perf_counter() - t)
            else:
                invalid.append((i, p_value))
            kkt["SI fits"] += result['results'][0]['kkt_si']['fits']
            kkt["SI violations"] += result['results'][0]['kkt_si']['violations']
        else:
            n_empty += 1

        if i % 10 == 0:
            print(f"{i}/{n_rep}, "
                  f"p-values: {len(p_values)}, "
                  f"empty models: {n_empty}, "
                  f"invalid p-values: {len(invalid)}, "
                  f"time: {time.perf_counter() - start:.0f}s, "
                  f"KKT violations: observed {kkt['observed violations']}/{kkt['observed fits']}, "
                  f"SI {kkt['SI violations']}/{kkt['SI fits']}")

    return np.array(p_values), branches, n_empty

In [ ]:
list_p_value, branches, n_empty = run(n_rep)

In [ ]:
alpha = 0.05
FPR = np.mean(list_p_value <= alpha)
print(f"FPR = {FPR}")

In [ ]:
statistic, p_value = stats.kstest(list_p_value, 'uniform')
print(f'KS-Test statistic = {statistic}')
print(f'KS-Test p-value = {p_value}')

In [ ]:
plt.hist(list_p_value, bins=20)

In [ ]:
plt.rcParams.update({'font.size': 18})
grid = np.linspace(0, 1, 101)
ecdf = np.searchsorted(np.sort(list_p_value), grid, side='right') / len(list_p_value)
plt.plot(grid, ecdf, 'r-', linewidth=6, label='p_value')
plt.plot([0, 1], [0, 1], 'k--')
plt.legend()
plt.tight_layout()
plt.show()